# Тема 10. Архитектуры ролевого управления: иерархическая МАС

**Модуль III · Пример 3 из 5**

Четыре базовые архитектуры МАС:

| Архитектура | Идея | Предсказуемость / отладка | Отказоустойчивость / масштаб |
|-------------|------|---------------------------|-------------------------------|
| **Иерархическая** | координатор и подчинённые специалисты | высокая | единая точка отказа |
| **Децентрализованная** | равноправные агенты, прямое общение | низкая | высокие |
| **Централизованная (звезда)** | роутер направляет запрос одному агенту | высокая | ограниченный масштаб |
| **Общий пул сообщений** | шина «издатель–подписчик» | средняя | асинхронность, масштаб |

Основное внимание — **иерархической** архитектуре (самая распространённая на практике): планировщик → специалисты с ограниченными инструментами → координатор → критик. В конце — краткая демонстрация **роутера-звезды**.

In [1]:
import os, json
from openai import OpenAI
from pydantic import BaseModel, ValidationError
import mas_domain as dom

BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:8000/v1")
API_KEY  = os.environ.get("LLM_API_KEY", "")
MODEL    = os.environ.get("LLM_MODEL", "qwen/qwen3.7-max")
client = OpenAI(api_key=API_KEY or "EMPTY", base_url=BASE_URL)
print("Готово:", MODEL)

Готово: qwen/qwen3.7-max


## Специалисты с ограниченными наборами инструментов

Принцип **специализации**: каждый специалист видит только свои инструменты. Это снижает путаницу и повышает надёжность по сравнению с одним агентом, у которого все инструменты сразу.

In [2]:
SPECIALISTS = {
    "catalog_agent": ["search_products"],                 # поиск товаров
    "orders_agent":  ["get_order", "check_shipping"],     # заказы и доставка
    "policy_agent":  ["get_policy", "calc_refund"],       # политики и возвраты
}

def run_specialist(role: str, subtask: str, max_steps: int = 4) -> dict:
    """Мини агент-цикл специалиста с ОГРАНИЧЕННЫМ набором инструментов."""
    tools = dom.tools_subset(SPECIALISTS[role])
    messages = [
        {"role": "system", "content":
            f"Ты — специализированный агент '{role}' поддержки магазина. Реши подзадачу, "
            f"используя только свои инструменты. Каталог на английском."},
        {"role": "user", "content": subtask}]
    for _ in range(max_steps):
        resp = client.chat.completions.create(model=MODEL, messages=messages,
                                              tools=tools, max_tokens=500)
        msg = resp.choices[0].message
        messages.append(msg.model_dump())
        if not msg.tool_calls:
            return {"role": role, "status": "success", "result": msg.content}
        for tc in msg.tool_calls:
            out = dom.run_tool(tc.function.name, json.loads(tc.function.arguments))
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out, ensure_ascii=False)})
    return {"role": role, "status": "partial", "result": "лимит шагов"}

print(run_specialist("catalog_agent", "Найди USB-C hub и его цену")["result"][:120])

Вот что я нашёл:

**USB-C Hub**
- **SKU:** P-300
- **Цена:** 2 490 ₽
- **В наличии:** 15 шт.

Если нужна дополнительная 


## Планировщик: структурированный план (structured output)

Планировщик декомпозирует запрос в **структурированный план**: список подзадач с назначенной ролью и приоритетом. Просим модель вернуть JSON и валидируем схемой Pydantic (при сбое — один повтор).

In [3]:
class SubTask(BaseModel):
    agent_role: str      # одна из ролей SPECIALISTS
    description: str
    priority: int

class Plan(BaseModel):
    subtasks: list[SubTask]

def make_plan(user_query: str) -> Plan:
    roles = list(SPECIALISTS)
    prompt = (f"Разбей запрос пользователя на подзадачи для специалистов {roles}. "
              f"Верни ТОЛЬКО JSON вида "
              f'{{"subtasks":[{{"agent_role":"...","description":"...","priority":1}}]}}.\n\n'
              f"Запрос: {user_query}")
    for attempt in range(2):
        raw = client.chat.completions.create(model=MODEL, max_tokens=500,
            messages=[{"role": "user", "content": prompt}]).choices[0].message.content
        txt = raw[raw.find("{"): raw.rfind("}") + 1]
        try:
            plan = Plan.model_validate_json(txt)
            plan.subtasks = [s for s in plan.subtasks if s.agent_role in SPECIALISTS]
            if plan.subtasks:
                return plan
        except (ValidationError, ValueError):
            pass
    return Plan(subtasks=[SubTask(agent_role=r, description=user_query, priority=1) for r in roles])

QUERY = ("Хочу вернуть заказ ORD-1001, посчитай сумму возврата с учётом политики, "
         "и подскажи, есть ли в наличии USB-C хаб и когда приедет заказ ORD-1002.")
plan = make_plan(QUERY)
for s in plan.subtasks:
    print(f"  [{s.priority}] {s.agent_role}: {s.description[:70]}")

  [1] orders_agent: Получить детали заказов ORD-1001 (состав и стоимость для возврата) и O
  [2] policy_agent: Рассчитать итоговую сумму возврата для заказа ORD-1001 с учётом действ
  [3] catalog_agent: Найти USB-C хаб в каталоге и проверить его текущее наличие.


## Координатор: делегирование и агрегация

Координатор исполняет план — раздаёт подзадачи специалистам (по возрастанию приоритета) и **агрегирует** результаты в единый ответ. Специалисты не общаются напрямую — только через координатора (иерархия).

In [4]:
def coordinator(user_query: str, verbose: bool = True) -> dict:
    plan = make_plan(user_query)
    results = []
    for s in sorted(plan.subtasks, key=lambda x: x.priority):
        r = run_specialist(s.agent_role, s.description)
        if verbose:
            print(f"  {s.agent_role} [{r['status']}]: {str(r['result'])[:80]}")
        results.append(r)
    joined = "\n".join(f"[{r['role']}] {r['result']}" for r in results)
    final = client.chat.completions.create(model=MODEL, max_tokens=500,
        messages=[{"role": "system", "content":
            "Ты координатор. Собери из результатов специалистов единый полный ответ, "
            "покрывающий ВСЕ части запроса."},
            {"role": "user", "content": f"Запрос: {user_query}\n\nРезультаты:\n{joined}"}]).choices[0].message.content
    return {"answer": final, "results": results}

out = coordinator(QUERY)
print("\n=== Ответ иерархической МАС ===\n", out["answer"][:600])

  orders_agent [success]: ## Информация по заказу ORD-1001

Я получил следующие данные:

| Параметр | Знач


  policy_agent [success]: ## Результат расчёта возврата для заказа ORD-1001

### 📋 Статус заказа
| Парамет


  catalog_agent [success]: ## Результат проверки наличия

Товар **USB-C Hub** найден в каталоге!

| Парамет


  orders_agent [success]: Вот информация о заказе **ORD-1002**:

### 📦 Заказ
| Параметр | Значение |
|----



=== Ответ иерархической МАС ===
 ## Ответ на ваш запрос

### 1️⃣ Возврат заказа ORD-1001

✅ **Возврат возможен!** Ваш заказ ORD-1001 (товар P-100, 1 шт.) был доставлен 01.07.2026 и находится в пределах 14-дневного срока возврата.

**💰 Сумма возврата: 4 990 ₽**
- Комиссия за возврат: 0 ₽
- Итого к возврату: **4 990 рублей** (полная стоимость заказа)

⚠️ **Для оформления возврата** необходимо обратиться к специализированному сервису возвратов, так как у меня нет прямого доступа к системе оформления. Передайте следующие данные:
- Заказ: ORD-1001
- Товар: P-100 (1 шт.)
- Сумма: 4 990 ₽

---

### 2️⃣ Наличие USB-C хаба

✅ **Товар 


## Критик: верификация (принцип рефлексии)

Критик проверяет агрегированный ответ на полноту и соответствие политикам, ставит оценку и при `score < 7` инициирует доработку. Важно: **модель критика не слабее** оцениваемых агентов, иначе он выдумывает несуществующие проблемы.

In [5]:
class CriticFeedback(BaseModel):
    approved: bool
    score: float
    issues: list[str]

def critic(user_query: str, answer: str) -> CriticFeedback:
    prompt = (f"Оцени ответ ассистента на запрос по полноте и соответствию политикам магазина. "
              f'Верни ТОЛЬКО JSON: {{"approved":true/false,"score":0-10,"issues":["..."]}}.\n\n'
              f"Запрос: {user_query}\n\nОтвет: {answer}")
    raw = client.chat.completions.create(model=MODEL, max_tokens=300,
        messages=[{"role": "user", "content": prompt}]).choices[0].message.content
    try:
        return CriticFeedback.model_validate_json(raw[raw.find("{"): raw.rfind("}") + 1])
    except (ValidationError, ValueError):
        return CriticFeedback(approved=True, score=7.0, issues=[])

def mas_with_critic(user_query: str) -> str:
    out = coordinator(user_query, verbose=False)
    fb = critic(user_query, out["answer"])
    print(f"Критик: approved={fb.approved}, score={fb.score}, issues={fb.issues[:2]}")
    if fb.score >= 7 and fb.approved:
        return out["answer"]
    # доработка с учётом замечаний критика
    revised = client.chat.completions.create(model=MODEL, max_tokens=500,
        messages=[{"role": "system", "content": "Ты координатор. Исправь ответ по замечаниям критика."},
                  {"role": "user", "content":
                      f"Запрос: {user_query}\nОтвет: {out['answer']}\nЗамечания: {fb.issues}"}]
        ).choices[0].message.content
    return revised

print("\n=== МАС + критик ===\n", mas_with_critic(QUERY)[:600])

Критик: approved=False, score=5.0, issues=['Ассистент не имеет доступа к базам заказов/остатков и не может подтверждать точные суммы, остатки (15 шт), сроки (2 дня) и перевозчика (СДЭК) — высока вероятность галлюцинаций.', 'Заявление об «обработке запросов профильными специалистами» некорректно и вводит пользователя в заблуждение о возможностях ассистента.']



=== МАС + критик ===
 Здравствуйте! Я виртуальный ассистент, и у меня нет прямого доступа к базам данных заказов, складским остаткам и логистическим службам в реальном времени. Поэтому я не могу назвать точные суммы, сроки или подтвердить наличие товара. 

Однако я подробно расскажу, как работают наши правила, и подскажу, где посмотреть нужную информацию.

### 1. ↩️ Возврат заказа ORD-1001
Согласно политике магазина, возврат возможен при соблюдении следующих условий:
* С момента доставки прошло **не более 14 дней**.
* Товар **не имеет следов использования**, сохранены его товарный вид, пломбы и оригинальная упаковк


## Централизованная архитектура (звезда): роутер

Для сравнения — **роутер-диспетчер**: он не декомпозирует задачу, а классифицирует запрос и направляет его **одному** подходящему специалисту. Прост и предсказуем для чётко классифицируемых запросов, но плохо покрывает многодоменные.

In [6]:
def router(user_query: str) -> dict:
    role = client.chat.completions.create(model=MODEL, max_tokens=10, temperature=0,
        messages=[{"role": "system", "content":
            "Ты роутер. Верни ОДНУ роль из списка: catalog_agent, orders_agent, policy_agent — "
            "наиболее подходящую под запрос. Только имя роли."},
            {"role": "user", "content": user_query}]).choices[0].message.content.strip()
    role = next((r for r in SPECIALISTS if r in role), "orders_agent")
    print(f"Роутер направил запрос -> {role}")
    return run_specialist(role, user_query)

print(router("Какой статус доставки у заказа ORD-1002?")["result"][:200])

Роутер направил запрос -> orders_agent


Статус доставки заказа **ORD-1002**:

- **Состояние:** 🚚 **В пути** (in_transit)
- **Перевозчик:** CDEK
- **Ориентировочный срок доставки:** 2 дня

Ваш заказ уже в пути и должен прибыть в ближайшие 2 


## Итоги

- **Иерархическая** архитектура: планировщик (структурированный план) → специалисты с **ограниченными** инструментами → координатор-агрегатор → критик. Явная декомпозиция **гарантирует покрытие** всех доменов запроса.
- **Критик** служит сеткой безопасности; его модель должна быть не слабее оцениваемых агентов.
- **Звезда (роутер)** проще, но лишь перенаправляет запрос и не годится для многодоменных задач.
- Плата за МАС — больше вызовов модели и задержка; выбор архитектуры — под задачу.

**Дальше:** Тема 11 — контракты сообщений между агентами (A2A) и разграничение MCP/A2A.